# Step 4 — word2vec on the address text

**Why:** ESIM reads words, not characters, and its word vectors come from a word2vec model trained on our own address text (Step 4 spec, 2026-10-24: vector size 200, skip-gram, all splits' text from both sides, no labels, frozen in ESIM).

**Open settings:** `min_count`, `window`, epochs; the nearest-neighbour check goes in Diagnostics.

## 0. Setup
**Why:** one place for imports and the dataset name every cell uses.

In [2]:
# Setup: imports and dataset name
%load_ext autoreload
%autoreload 2

import re
import sys
from collections import Counter
sys.path.append("..")   # geocoding_ml/ -> lets us import from helpers/

import numpy as np
import pandas as pd
from gensim.models import Word2Vec, FastText
from sklearn.metrics import roc_auc_score

from helpers.pairs import load_pairs
from esim.tokenizer import tokenize   # shared with 02_esim.ipynb

pd.set_option("display.width", 250)
pd.set_option("display.expand_frame_repr", False)

DATASET = "overture_case2s"   # decided 2026-10-30

## 1. Load the pairs

In [3]:
# Load: train / val / test pairs
train, train_label = load_pairs(DATASET, "train")
val, val_label = load_pairs(DATASET, "val")
test, test_label = load_pairs(DATASET, "test")
full_dataset = pd.concat([train, val, test])
pd.concat([train, val, test]).groupby(["split", "neg_type"], observed = True).size().unstack()

neg_type,case0,case1,case2,case2s,case3
split,,,,,
train,20259,67174,172,5441,109522
val,4187,13781,34,894,22969
test,4386,14448,15,1053,23942


### Generating Corpus
- Combine all addresses from `address_a` and `address_b`
- Remove `address` dupes, because they can appear multiple times dataset. For example, can exist in pos and neg
- Dedupe on id, to just avoid row duplication

In [4]:
id_and_address = pd.concat([
    full_dataset[["address_a", "id_a"]].rename(columns = {
        "address_a":"address",
        "id_a": "id",
    }),
    full_dataset[["address_b", "id_b"]].rename(columns = {
        "address_b": "address",
        "id_b": "id"
    })
]).drop_duplicates(subset = ["id", "address"])["address"]


### Checking the value counts
- We want to do a check on the corpus and see how the distribution word looks for each frequency amount
- For example, what words are one-ofs vs multiple occurences. Are they typos, or maybe numbers?

In [5]:
#Creates a Series as a Counter() with char and frequency
word_counts = id_and_address.str.split().explode().value_counts()
#Creates series where frequency is replaced w/ a label
word_counts_cutoff = pd.cut(word_counts, bins = [0,1,4,9,np.inf], labels = ["1","2-4", "5-9", ">10"])

#Total statistical check
id_and_address.str.split().str.len().describe()

# Checks distribution of frequency amount
for label in ["1","2-4", "5-9", ">10"]:
    print(f"{label}: {len(word_counts_cutoff[word_counts_cutoff == label])}")

# Plaintext list of items
for word_cutoff in ["1", "2-4"]:
    print(word_counts_cutoff[
        (word_counts_cutoff == word_cutoff) & 
        (~word_counts_cutoff.index.str.match(r"[\d-]+"))].sample(frac=1).head(10))

1: 2270
2-4: 1783
5-9: 1010
>10: 1602
address
SNIPSIC       1
SHAILOR       1
SHEEPSKIN     1
LORIA         1
ICHABOD       1
MOODYS        1
PATTONWOOD    1
TAHMORE       1
EMILY         1
MEAT          1
Name: count, dtype: category
Categories (4, object): ['1' < '2-4' < '5-9' < '>10']
address
BENNETT        2-4
ORE            2-4
VANBLOCK       2-4
SWAN           2-4
TAUTOG         2-4
CANTOR         2-4
AVALON         2-4
NORWICHTOWN    2-4
REGAN          2-4
ALVCORD        2-4
Name: count, dtype: category
Categories (4, object): ['1' < '2-4' < '5-9' < '>10']


### 2. word2vec: choose the epoch count
- **Tokenize**: returns a sequence of words/numbers and then upper-cases them

- **Address_vector**: iterates over all tokens using model embeddings. 
    - all unknown embeddings get ignored
    - keep model safe by never inputting an empty list i.e. `if not known`
    - Then: actually run embedding, and squash all word embeddings in sentence into one

- **Pair Cosines**: 
    - Runs a dot product between `address_a` and `address_b`. If the model is correct, we should expect their similarity to be high

- **AUC**
    - We generate a `pair_cosines`dataset with appropriate labels
    - Running `roc_auc` will tests and give a threshold distribution, of how different thresholds do in correctly matching all neg/pos pairs

In [6]:
# word2vec helpers: tokenize, address vector, pair cosines, AUC
# 1 tokenize: moved to esim/tokenizer.py (imported in Setup), so 02_esim.ipynb uses the same one
# 2
def address_vector(model, tokens):
    known = [t for t in tokens if t in model.wv.key_to_index]
    if not known:
        return np.zeros(model.vector_size)
    return model.wv[known].mean(axis = 0)
# 3
def pair_cosines(model, df):
    va = np.stack([address_vector(model, tokenize(s)) for s in df["address_a"]])
    vb = np.stack([address_vector(model, tokenize(s)) for s in df["address_b"]])
    return (va * vb).sum(axis = 1) / (np.linalg.norm(va, axis =1) * np.linalg.norm(vb, axis =1) + 1e-9)

def auc(model, pos, neg):
    labels = np.concatenate([np.ones(len(pos)), np.zeros(len(neg))])
    scores = np.concatenate([
        pair_cosines(model, pos),
        pair_cosines(model, neg)
    ])
    return roc_auc_score(labels, scores)

### Cell B — corpus, val slices, epoch loop (puzzle: 9 slots)
- We are running word2vec unsupervised on embedded sentences, for varying amount of epochs
- We are evaluating by isolating on `neg_type`
- We are evaluating auc using `case3/case2s` for negatives and `case0` for all positive cases
    - We evaluate AUC by first matching `pos_pairs` with `neg_pairs`. Each will have their cosine_distance. If `pos_distance` > `neg_distance` that's 1 (opposite is 0). AUC is the percentage, for which this occurs.

In [7]:
# Epoch choice: train at 5 / 20 / 50 epochs, val AUC per setting

sentences = id_and_address.map(tokenize).tolist()
results = {}

val_case2s = val[val["neg_type"] == "case2s"]
val_case3 = val[val["neg_type"] == "case3"]
val_pos = val[(val["neg_type"] == "case0") & (val["address_a"] != val["address_b"])]

for min_count in [1, 2]:
    for epochs in [5, 20, 50]:
        model = Word2Vec(sentences, vector_size=200, sg=1, window = 2, min_count=min_count, epochs= epochs, seed=0, workers=1)
        
        results[epochs] = {"case2s": auc(model, val_pos, val_case2s), "case3": auc(model, val_pos, val_case3)}
    assert model.epochs == 50
    model.save(f"checkpoints/word2vec_50_mc{min_count}.model")
    display(pd.DataFrame(results).T)
    
# Save the 50-epoch model for 02_esim.ipynb
    


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


,case2s,case3
5,0.604523,0.909628
20,0.592902,0.942507
50,0.614408,0.966185


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


,case2s,case3
5,0.600072,0.907861
20,0.587032,0.942498
50,0.619810,0.965846


### 3. fastText variant (2026-11-05, set up by Claude at David's request)
- Same settings as the chosen word2vec (skip-gram, 200-d, `window=2`, 50 epochs, `min_count=1`), plus character n-grams of length 3–6 (`min_n`, `max_n`, gensim defaults)
- A word's vector = its own vector + the vectors of its n-grams, so `RAOD` lands near `ROAD`; with `min_count=1` every corpus word is in the vocabulary, so ESIM uses the table exactly like word2vec's
- Saved as `checkpoints/fasttext_50_mc1.model`; `02_esim.ipynb` loads it through the word2vec switch


In [8]:
# fastText: same settings as word2vec min_count=1, plus char n-grams 3-6
# Self-contained (needs Setup, Load, corpus, helpers), so the 6-model epoch loop above need not be rerun
sentences = id_and_address.map(tokenize).tolist()
val_case2s = val[val["neg_type"] == "case2s"]
val_case3 = val[val["neg_type"] == "case3"]
val_pos = val[(val["neg_type"] == "case0") & (val["address_a"] != val["address_b"])]

ft = FastText(sentences, vector_size=200, sg=1, window=2, min_count=1, epochs=50, min_n=3, max_n=6, seed=0, workers=1)
print({"case2s": auc(ft, val_pos, val_case2s), "case3": auc(ft, val_pos, val_case3)})   # same val AUCs as the word2vec table
ft.save("checkpoints/fasttext_50_mc1.model")


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


{'case2s': np.float64(0.6229211051510277), 'case3': np.float64(0.9694808227968382)}


### Results
- `case2` seems to have limited ability to dillineate for `word2vec`. Result of 0.6 indicates that it's not very easily tell the difference between positive & negative pairs. Though, this was sort of to be expected, since we intentionally made them hard examples that were contextually similar
    - Since it doesn't seem to improve with epochs, we can tenatively conclude that word2vec has hit its limit
    - **Why does case2s seem to fluctuate?**: It fluctuates likely because of noise. These are fairly minimal adjustments that can be explained by different seeds during unsupervised learning
    - **Why does case2s not improve with more epochs?**: Because, word2vec is optimized on predicting its neighbors, not necessarily being able to reduce cosine-distance between positive and negative pairs
- `case3` seems to learn quite well. It has a high cosine-distance between positive and negative pairs, which indicates in a sizeable AUC score. This makes sense, because `case3` was made with different random addresses, meaning they have a lot of different context to tell the difference. Additionally, the fact that it seems to improve w/ epochs seems to indicate `word2vec` is highly suited to differentiate words within this case study.

## Diagnostics
One-off inspections, kept out of the pipeline cells above. Results are written into the markdown (question → result → query).

Gut-check if APT really didn't show up as much as it did
- Reason seems likely due to `Overture's` methodology of only trying to mark `area_of_interest` and NOT RESIDENCES

In [9]:
from collections import Counter
counts = Counter(w for s in id_and_address for w in tokenize(s))   # in 01_word2vec, after the corpus cell
{w: counts[w] for w in ["APT", "STE", "SUITE", "UNIT"]}

{'APT': 1, 'STE': 105, 'SUITE': 40, 'UNIT': 38}

Checking if there's an correlation between the cosine-distance and difference between two number embeddings
- There shouldn't be a real relationship. 1) Because the model has many one-ofs, so any correlation would be weak
- Additionally, house_numbers are not compared to each other, rather they are made in context of the total address, so they shouldn't really be related to begin with
- **Result**: no clear correlation between the two, expected result

In [10]:
import random
w2v = Word2Vec.load("checkpoints/word2vec_50_mc1.model")

NUM_ROWS_TEST = 20
nums = [random.sample([k for k in w2v.wv.key_to_index if k.isdigit()], k=2) for _ in range(NUM_ROWS_TEST)]
df = pd.DataFrame(
    [{"address_a":num1, "address_b":num2}
    for num1, num2 in nums]
)
df["difference"] = (df["address_a"].astype(int) - df["address_b"].astype(int)).abs()

df["similarity"] = df.apply(lambda row: w2v.wv.similarity(row["address_a"], row["address_b"]), axis = 1)

display(df.sort_values("difference", ascending= False))


,address_a,address_b,difference,similarity
18,279,6900,6621,0.558531
2,1900,06331,4431,0.596890
1,647,4531,3884,0.598334
14,1200,4798,3598,0.696167
11,1010,3575,2565,0.589694
17,291,2162,1871,0.434884
3,2204,566,1638,0.473133
9,3029,1455,1574,0.648647
4,3557,2000,1557,0.645863
5,1745,2985,1240,0.679104


We have concerns that for `word2vec` where we lower `min_count=2 -> min_count=1`, our new numbers will not meaningfully be able to tell the difference between different and similar numbers.
- This test proves the opposite, since we can see numbers which are closely related, also have a high`cosine_similarity`
- **But Wouldn't we be checking a number against itself?**
    - When building the row, we deduplicate on `id, address`. Wouldn't that case issues?
    - No, since we're checking on neg1, this issue won't exist since it'll always compare on different id, and house_number (by construction)

In [11]:
# min_count=1 collision check: case1 UNK-group pairs vs random house-number pairs (written by Claude 2026-11-05)
old_vocab = Word2Vec.load("checkpoints/word2vec_50_mc2.model").wv.key_to_index   # the vocab ESIM was trained with
new_wv = Word2Vec.load("checkpoints/word2vec_50_mc1.model").wv

# Group: val case1 rows, both first tokens are numbers unknown to the old vocab, rest of the tokens identical
group_pairs = []
for a, b in zip(val.loc[val["neg_type"] == "case1", "address_a"], val.loc[val["neg_type"] == "case1", "address_b"]):
    ta, tb = tokenize(a), tokenize(b)
    if (ta and tb and ta[0].isdigit() and tb[0].isdigit()
            and ta[0] not in old_vocab and tb[0] not in old_vocab and ta[1:] == tb[1:]
            and ta[0] != tb[0]):   # two different numbers: never a word compared with itself
        group_pairs.append((ta[0], tb[0]))
assert len(group_pairs) == 161, len(group_pairs)   # same group as 02_esim D3

# Baseline: random pairs of house numbers (digits only, no leading 0 -> drops ZIP-like tokens)
house_numbers = [k for k in new_wv.key_to_index if k.isdigit() and not k.startswith("0")]
rng = random.Random(0)
random_pairs = [tuple(rng.sample(house_numbers, 2)) for _ in range(1000)]

sims = pd.DataFrame({
    "case1 UNK group (n=161)": pd.Series([new_wv.similarity(a, b) for a, b in group_pairs]),
    "random numbers (n=1000)": pd.Series([new_wv.similarity(a, b) for a, b in random_pairs]),
})
display(sims.describe().T)
print("share with similarity > 0.9:")
print(sims.apply(lambda col: (col.dropna() > 0.9).mean()))   # dropna: the group column is shorter (NaN-padded)


,count,mean,std,min,25%,50%,75%,max
case1 UNK group (n=161),161.0,0.924758,0.045076,0.813337,0.888649,0.917416,0.971868,0.990355
random numbers (n=1000),1000.0,0.568746,0.144347,0.074999,0.475316,0.581036,0.666022,0.967113


share with similarity > 0.9:
case1 UNK group (n=161)    0.664596
random numbers (n=1000)    0.006000
dtype: float64


### Same collision check for fastText (Claude, 2026-11-05)
- Same 161 group pairs and 1,000 random pairs as the cell above, fastText vectors instead
- `4817` / `4819` share n-grams (`<48`, `481`, `<481`), so some similarity is expected; the question is whether the group still sits far above random
- **Result:** _pending_


In [12]:
# fastText collision check: reuses group_pairs / random_pairs from the cell above
ft_wv = FastText.load("checkpoints/fasttext_50_mc1.model").wv
ft_sims = pd.DataFrame({
    "case1 UNK group (n=161)": pd.Series([ft_wv.similarity(a, b) for a, b in group_pairs]),
    "random numbers (n=1000)": pd.Series([ft_wv.similarity(a, b) for a, b in random_pairs]),
})
display(ft_sims.describe().T)
print("share with similarity > 0.9:")
print(ft_sims.apply(lambda col: (col.dropna() > 0.9).mean()))


,count,mean,std,min,25%,50%,75%,max
case1 UNK group (n=161),161.0,0.787193,0.046772,0.627971,0.761865,0.787080,0.811809,0.924297
random numbers (n=1000),1000.0,0.677946,0.102615,0.226880,0.632704,0.693126,0.744956,0.919239


share with similarity > 0.9:
case1 UNK group (n=161)    0.018634
random numbers (n=1000)    0.004000
dtype: float64
